In [40]:
pip install keras-image-helper

Note: you may need to restart the kernel to use updated packages.


In [47]:
# this is just the predict out of the full library
!pip install --extra-index-url https://google-coral.github.io/py-repo/ tflite_runtime

Looking in indexes: https://pypi.org/simple, https://google-coral.github.io/py-repo/


In [50]:
from PIL import Image
#import tensorflow.lite as tflite
from keras_image_helper import create_preprocessor

In [24]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img, ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import tensorflow.lite as tflite
tf.__version__

'2.8.0'

In [3]:
model = keras.models.load_model('MobileNetsV2.04-0.86.h5')

In [8]:
img = load_img('pants.jpg', target_size=(224,224))

In [11]:
X = preprocess_input(np.array([np.array(img)]))

In [15]:
pred = model.predict(X)

In [16]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

In [21]:
dict(zip(classes, pred[0]))

{'dress': -0.65221864,
 'hat': -2.267601,
 'longsleeve': 0.4639197,
 'outwear': 1.8182472,
 'pants': 7.1153307,
 'shirt': -1.1791565,
 'shoes': -2.0582182,
 'shorts': 1.0777522,
 'skirt': -0.5480293,
 't-shirt': -2.9732924}

In [22]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tf_lite_model = converter.convert()

with open('clothing-model.tflite', 'wb') as f:
    f.write(tf_lite_model)

INFO:tensorflow:Assets written to: C:\Users\alaa_\AppData\Local\Temp\tmps7ca1nj2\assets


In [51]:
interpet = tflite.Interpreter(model_path='clothing-model.tflite')
interpet.allocate_tensors()

In [33]:
input_index = interpet.get_input_details()[0]['index']

output_index = interpet.get_output_details()[0]['index']

In [38]:
interpet.set_tensor(input_index, X)

interpet.invoke()

preds = interpet.get_tensor(output_index)

In [37]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]
dict(zip(classes, pred[0]))

{'dress': -0.65221864,
 'hat': -2.267601,
 'longsleeve': 0.4639197,
 'outwear': 1.8182472,
 'pants': 7.1153307,
 'shirt': -1.1791565,
 'shoes': -2.0582182,
 'shorts': 1.0777522,
 'skirt': -0.5480293,
 't-shirt': -2.9732924}

In [58]:
with Image.open('pants.jpg') as img:
    img = img.resize((224, 224), Image.NEAREST)

In [59]:
def preprocess_input(x):
    x /= 127.5
    x -= 1.
    return x

In [60]:
x = np.array(img, dtype='float32')
X = np.array([x])

X = preprocess_input(X)

In [61]:
interpet.set_tensor(input_index, X)
interpet.invoke()
preds = interpet.get_tensor(output_index)

In [63]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

dict(zip(classes, preds[0]))

{'dress': -0.65222055,
 'hat': -2.2675984,
 'longsleeve': 0.46392,
 'outwear': 1.8182452,
 'pants': 7.1153307,
 'shirt': -1.1791563,
 'shoes': -2.0582192,
 'shorts': 1.0777485,
 'skirt': -0.5480271,
 't-shirt': -2.9732974}